In [6]:
#ss_conection_test_v0.1.json

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import smartsheet
import pandas as pd
import json

CONFIG_PATH = os.path.join("..","..","..","config.json")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

def cargar_dataframe(sheet_id: str):
    token = config["SS_Assistants"]["Smartsheet"]
    if not token:
        raise RuntimeError("Define SMARTSHEET_TOKEN en el entorno")
    client = smartsheet.Smartsheet(token)
    sheet = client.Sheets.get_sheet(sheet_id)
    cols = [c.title for c in sheet.columns]
    data = []
    for row in sheet.rows:
        row_data = {col: cell.value for col, cell in zip(cols, row.cells)}
        data.append(row_data)
    return pd.DataFrame(data)

df = cargar_dataframe('GhQRHWJvqh9P33wf87FRW75cRf3WFpPv3cGC3vq1')
df = df.tail(5)

In [7]:
print(df.head())

       A Nivel  Consecutivo Estado / Prioridad  \
90  None  None  UNG-ODPA065               pago   
91  None  None  UNG-ODPA066               Pago   
92  None  None  UNG-ODPA067               Pago   
93  None  None  UNG-ODPA068               Pago   
94  None  None  UNG-ODPA069               Pago   

                                             Concepto  \
90   SOLICITUD DE PAGO DE PLANILLA ARL - FEBRERO 2023   
91  PAGO POR REVISION DE DISEÑOS HIDRAULICOS, SANI...   
92                                REEMBOLSO DE GASTOS   
93                                   PAGO IVA P1 2023   
94  SERVICIOS PRESTADOS DICIEMBRE-2023_CLAUDIA BOR...   

                  Presupuesto Din / Tercero 13 Anticipos  / No. Factura-DE  \
90  Caja de compensación familiar COMPENSAR                     66933801.0   
91            PAOLA CRISTINA RAMIREZ GUZMAN                         CC 001   
92                             DAVID BLANCO                    UNG-CJM-004   
93                                     DIAN 

In [8]:
import openai
api_key = config["SS_Assistants"]["OPENAI"]
client = openai.OpenAI(api_key=api_key)

In [9]:
def answer_with_openai(df, question):
    # Contexto limpio: primeros 50 valores de “Concepto”

    prompt = (
        f"Con los datos de la tabla {df}:\n"
        "Responde la siguiente pregunta:"

        f"Pregunta: {question}\n"
        "Por favor, responde concisamente en español."
    )
    print("DEBUG prompt:\n ", prompt)

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role":"system","content":"Eres un asistente que responde sobre un proyecto."},
            {"role":"user","content":prompt}
        ],
        temperature=0.1
    )
    return resp.choices[0].message.content.strip()

In [10]:

while True:
    q = input("Pregunta ('salir' para terminar): ")
    if q.lower() in ("salir","exit"): break
    print("\nRespuesta:", answer_with_openai(df, q))




DEBUG prompt:
  Con los datos de la tabla        A Nivel  Consecutivo Estado / Prioridad  \
90  None  None  UNG-ODPA065               pago   
91  None  None  UNG-ODPA066               Pago   
92  None  None  UNG-ODPA067               Pago   
93  None  None  UNG-ODPA068               Pago   
94  None  None  UNG-ODPA069               Pago   

                                             Concepto  \
90   SOLICITUD DE PAGO DE PLANILLA ARL - FEBRERO 2023   
91  PAGO POR REVISION DE DISEÑOS HIDRAULICOS, SANI...   
92                                REEMBOLSO DE GASTOS   
93                                   PAGO IVA P1 2023   
94  SERVICIOS PRESTADOS DICIEMBRE-2023_CLAUDIA BOR...   

                  Presupuesto Din / Tercero 13 Anticipos  / No. Factura-DE  \
90  Caja de compensación familiar COMPENSAR                     66933801.0   
91            PAOLA CRISTINA RAMIREZ GUZMAN                         CC 001   
92                             DAVID BLANCO                    UNG-CJM-004   
93